In [25]:
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt

In [26]:
# ==============================================================================
# "TUNING" SECTION
# ==============================================================================
# --- SCRIPT MODE ---
# Set this to match the data you want to visualize (Dummy or Real)
USE_DUMMY_DATA = True

# --- FILE & FOLDER NAMING ---
# IMPORTANT: These MUST EXACTLY match the variables in your main analysis script!
SEGMENT_FOLDER_NAME = "Sections"
SEGMENT_FILENAME_PREFIX = "sections_"

In [27]:
# Directories and paths

script_dir = os.getcwd()
if USE_DUMMY_DATA:
    telemetry_path = os.path.join(script_dir, 'Telemetry', 'Dummy')
    segments_path = os.path.join(script_dir, SEGMENT_FOLDER_NAME, 'Dummy')
    plot_output_path = os.path.join(script_dir, 'Plots', 'Dummy')
else:
    telemetry_path = os.path.join(script_dir, 'Telemetry', 'Real')
    segments_path = os.path.join(script_dir, SEGMENT_FOLDER_NAME, 'Real')
    plot_output_path = os.path.join(script_dir, 'Plots', 'Real')

os.makedirs(plot_output_path, exist_ok=True)
print(f"Plot output folder is '{plot_output_path}'")

# --- Step 2: FIND ALL SEGMENT FILES TO VISUALIZE ---
# Use the tunable variables to build the search path
segment_files_pattern = os.path.join(segments_path, f'{SEGMENT_FILENAME_PREFIX}*.csv')
segment_files = glob.glob(segment_files_pattern)

if not segment_files:
    print(f"No segment files found in '{segments_path}'. Please run the main analysis script first.")
    exit()
print(f"Found {len(segment_files)} segment files to visualize.")

Plot output folder is 'h:\My Drive\Cranfield\11. Individual Thesis\Code\Plots\Dummy'
Found 7 segment files to visualize.


In [28]:
# Execution

overwrite_all = None
for segment_file in segment_files:
    # Extract track name using the tunable prefix
    track_name = os.path.basename(segment_file).replace(SEGMENT_FILENAME_PREFIX, '').replace('.csv', '')
    print(f"\n--- Creating plot for track: {track_name.upper()} ---")

    try:
        segments_df = pd.read_csv(segment_file)
        telemetry_files_for_track = glob.glob(f'{telemetry_path}/{track_name}/**/*.csv', recursive=True)
        if not telemetry_files_for_track:
            raise FileNotFoundError(f"Could not find any telemetry files for track '{track_name}'.")
        sample_lap_df = pd.read_csv(telemetry_files_for_track[0])
        print(f"  Loaded segment data and sample telemetry lap: {os.path.basename(telemetry_files_for_track[0])}")
    except Exception as e:
        print(f"  An error occurred loading data for {track_name}: {e}. Skipping.")
        continue

    # --- Step 4: CREATE THE STACKED PLOT ---
    
    # Define which columns from the CSV we want to plot
    columns_to_plot = [
        'Car Speed', 'pBrake Front', 'rThrottle', 'aSteering', 
        'xDamperFL', 'xDamperFR', 'xDamperRL', 'xDamperRR'
    ]
    
    # Filter out columns that might not exist in the real data yet
    columns_to_plot = [col for col in columns_to_plot if col in sample_lap_df.columns]
    
    # Create a figure with a subplot for each channel. `sharex=True` is the key.
    fig, axes = plt.subplots(
        nrows=len(columns_to_plot), 
        ncols=1, 
        figsize=(20, 15), # (width, height) in inches
        sharex=True
    )
    fig.suptitle(f"Telemetry and Segments for {track_name.upper()}", fontsize=16, weight='bold')

    # Loop through each column and its corresponding subplot axis
    for i, column in enumerate(columns_to_plot):
        ax = axes[i]
        ax.plot(sample_lap_df['Distance'], sample_lap_df[column], label=column, color='navy')
        ax.set_ylabel(column, fontsize=10)
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.legend(loc='upper right')

        # Overlay the segments and boundary lines on each subplot
        for _, row in segments_df.iterrows():
            color = 'lightgrey' if row['segment_id'] % 2 != 0 else 'white'
            ax.axvspan(row['start_m'], row['end_m'], facecolor=color, alpha=0.3, zorder=-100)
            if row['start_m'] > 0:
                ax.axvline(x=row['start_m'], color='black', linestyle='--', linewidth=1)

    # Finalize the plot
    axes[-1].set_xlabel("Distance (m)", fontsize=12)
    plt.tight_layout(rect=[0, 0.03, 1, 0.97]) # Adjust layout to make room for suptitle

    # --- Step 5: SAVE THE PLOT (WITH OVERWRITE CHECK) ---
    output_filename = f"plot_{track_name}.png"
    full_output_path = os.path.join(plot_output_path, output_filename)

    if os.path.exists(full_output_path):
        if overwrite_all is None:
            choice = input("One or more plot files already exist. Overwrite ALL existing files? (y/n): ")
            overwrite_all = (choice.lower() == 'y')
        if overwrite_all:
            plt.savefig(full_output_path, dpi=150)
            print(f"  -> Plot overwritten: {full_output_path}")
        else:
            print(f"  -> Save skipped for existing file: {full_output_path}")
    else:
        plt.savefig(full_output_path, dpi=150)
        print(f"  -> Plot saved successfully to: {full_output_path}")
    plt.close(fig)

print("\n===== Visualization script finished. =====")


--- Creating plot for track: DONINGTON_GP ---
  Loaded segment data and sample telemetry lap: dummy_lap_1.csv
  -> Plot overwritten: h:\My Drive\Cranfield\11. Individual Thesis\Code\Plots\Dummy\plot_donington_gp.png

--- Creating plot for track: BRANDS_GP ---
  Loaded segment data and sample telemetry lap: dummy_lap_1.csv
  -> Plot overwritten: h:\My Drive\Cranfield\11. Individual Thesis\Code\Plots\Dummy\plot_brands_gp.png

--- Creating plot for track: HUNGARORING ---
  Loaded segment data and sample telemetry lap: dummy_lap_1.csv
  -> Plot overwritten: h:\My Drive\Cranfield\11. Individual Thesis\Code\Plots\Dummy\plot_hungaroring.png

--- Creating plot for track: MONZA ---
  Loaded segment data and sample telemetry lap: dummy_lap_1.csv
  -> Plot overwritten: h:\My Drive\Cranfield\11. Individual Thesis\Code\Plots\Dummy\plot_monza.png

--- Creating plot for track: SPA ---
  Loaded segment data and sample telemetry lap: dummy_lap_1.csv
  -> Plot overwritten: h:\My Drive\Cranfield\11. Ind